Imports

In [11]:
import pandas as pd
import numpy as np
import kagglehub
from umap import UMAP
from kagglehub import KaggleDatasetAdapter
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer
from langchain_google_genai import ChatGoogleGenerativeAI
from bertopic.representation import LangChain

Load Data

In [12]:
df = pd.read_csv('Data/finnhub_clean.csv')
df.head()

,stock,headline,date,date_only
0,JPM,"Inside ‘Project Eagle,’ JPMorgan’s High-Wire A...",2026-03-28 19:00:00,2026-03-28
1,JPM,"AI Schism Grips Washington as Tech, Labor Vie ...",2026-03-28 13:00:00,2026-03-28
2,JPM,Dip-buyers arrive to pull gold back from brink...,2026-03-28 09:00:52,2026-03-28
3,JPM,How The Standard Chartered (LSE:STAN) Story Is...,2026-03-28 00:15:29,2026-03-28
4,JPM,Epstein victims to get $72.5M from Bank of Ame...,2026-03-27 23:35:53,2026-03-27


In [13]:
# 1. Parse date and sort chronologically
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date', 'headline']).sort_values('date').reset_index(drop=True)

# 2. Define split proportions and embargo
SPLIT_TRAIN = 0.70
SPLIT_VAL = 0.15
SPLIT_TEST = 0.15
EMBARGO_N = 5  # Number of time-steps (e.g., hours/days) to drop to prevent leakage

# 3. Calculate date-based boundaries (keep same timestamp together)
unique_dates = np.array(sorted(df['date'].dt.floor('D').unique()))
n_dates = len(unique_dates)

train_end_idx = int(SPLIT_TRAIN * n_dates)
val_end_idx = int((SPLIT_TRAIN + SPLIT_VAL) * n_dates)

# 4. Create masks with embargo
day_code, _ = pd.factorize(df['date'].dt.floor('D'), sort=True)
mask_train = day_code < train_end_idx
mask_val = (day_code >= (train_end_idx + EMBARGO_N)) & (day_code < val_end_idx)
mask_test = day_code >= (val_end_idx + EMBARGO_N)

# 5. Extract splits
train_df = df[mask_train].copy()
val_df = df[mask_val].copy()
test_df = df[mask_test].copy()

# Final lists for the model
train_docs, train_timestamps = train_df['headline'].tolist(), train_df['date'].tolist()
val_docs, val_timestamps = val_df['headline'].tolist(), val_df['date'].tolist()
test_docs, test_timestamps = test_df['headline'].tolist(), test_df['date'].tolist()

print(f"Train size: {len(train_docs)}")
print(f"Validation size: {len(val_docs)} (Dropped {EMBARGO_N} day units for embargo)")
print(f"Test size: {len(test_docs)} (Dropped {EMBARGO_N} day units for embargo)")
print('Date ranges:')
print(f"  Train: {train_df['date'].min()} -> {train_df['date'].max()}")
print(f"  Val:   {val_df['date'].min()} -> {val_df['date'].max()}")
print(f"  Test:  {test_df['date'].min()} -> {test_df['date'].max()}")

Train size: 2248
Validation size: 607 (Dropped 5 day units for embargo)
Test size: 2003 (Dropped 5 day units for embargo)
Date ranges:
  Train: 2024-09-17 15:00:55 -> 2025-12-11 23:21:43
  Val:   2025-12-17 00:07:46 -> 2026-02-02 23:15:04
  Test:  2026-02-08 08:05:42 -> 2026-03-28 19:00:00


Prepare Models

In [14]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
train_embeddings = sentence_model.encode(train_docs, show_progress_bar=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1129.41it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1129.41it/s, Materializing param=pooler.dense.weight]                     
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 71/71 [00:03<00:00, 19.66it/s]


Train BERTopic

In [15]:
print("Training BERTopic model on training split...\n")

umap_model = UMAP(n_neighbors=15, n_components=10, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = OnlineCountVectorizer(stop_words="english")

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  calculate_probabilities=True,
  verbose=True
)

# Train BERTopic only on train split
topics_train, probs_train = topic_model.fit_transform(train_docs, embeddings=train_embeddings)

print("\nModel training complete!")

2026-03-29 01:15:34,841 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training BERTopic model on training split...



2026-03-29 01:15:38,823 - BERTopic - Dimensionality - Completed ✓
2026-03-29 01:15:38,824 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 01:15:38,824 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 01:15:39,095 - BERTopic - Cluster - Completed ✓
2026-03-29 01:15:39,097 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 01:15:39,095 - BERTopic - Cluster - Completed ✓
2026-03-29 01:15:39,097 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 01:15:39,133 - BERTopic - Representation - Completed ✓
2026-03-29 01:15:39,133 - BERTopic - Representation - Completed ✓



Model training complete!


Get Topic Information

In [16]:
# Get topic information
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,475,-1_asia_china_stock_q2,"[asia, china, stock, q2, credit, letter, week,...",[Major Asia-Pacific Banks Post Market Cap Gain...
1,0,220,0_canada_td_royal_toronto,"[canada, td, royal, toronto, dominion, bank, r...",[Royal Bank of Canada (RY) Q3 2025 Earnings Ca...
2,1,104,1_hsbc_holdings_hong_kong,"[hsbc, holdings, hong, kong, years, 52, invest...",[HSBC's Hong Kong Shares Dip After $13.6 Billi...
3,2,84,2_barclays_bcs_plc_reiterates,"[barclays, bcs, plc, reiterates, receipt, reco...",[Shore Capital Reiterates Barclays PLC - Depos...
4,3,73,3_societe_generale_société_générale,"[societe, generale, société, générale, voting,...",[Societe Generale: First quarter 2025 earnings...
5,4,72,4_commentary_fund_q1_international,"[commentary, fund, q1, international, 2025, q2...","[Harbor International Fund Q1 2025 Commentary,..."
6,5,71,5_ubs_ag_group_swiss,"[ubs, ag, group, swiss, rules, advisor, adviso...",[UBS Group AG Mulls U.S. Relocation as Swiss C...
7,6,67,6_bnp_paribas_stab_primary,"[bnp, paribas, stab, primary, issues, custody,...",[BNP Paribas Primary New Issues: STAB Notice -...
8,7,64,7_deutsche_db_bank_aktiengesellschaft,"[deutsche, db, bank, aktiengesellschaft, dbk, ...",[Deutsche Bank Aktiengesellschaft (DB) Q3 2025...
9,8,63,8_santander_banco_tsb_sabadell,"[santander, banco, tsb, sabadell, stake, san, ...","[Banco Santander, S.A. 2025 Q2 - Results - Ear..."


Visualize Topics

In [17]:
topic_model.visualize_documents(train_docs, embeddings=train_embeddings)

Preparing Validation

In [18]:
bertopic_topics = [
    [word for word, _ in topic_model.get_topic(t)]
    for t in topic_model.get_topics().keys() if t != -1
    if topic_model.get_topic(t) is not None
 ]

# Project validation documents into trained topic space
val_topics, val_probs = topic_model.transform(val_docs)

print("Validation transform complete (test set remains untouched).")

Batches: 100%|██████████| 19/19 [00:01<00:00, 17.70it/s]
2026-03-29 01:15:44,405 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
Batches: 100%|██████████| 19/19 [00:01<00:00, 17.70it/s]
2026-03-29 01:15:44,405 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-29 01:15:49,323 - BERTopic - Dimensionality - Completed ✓
2026-03-29 01:15:49,324 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-29 01:15:49,323 - BERTopic - Dimensionality - Completed ✓
2026-03-29 01:15:49,324 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-29 01:15:49,356 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-03-29 01:15:49,356 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-03-29 01:15:49,458 - BERTopic - Probabilities - Completed ✓
2026-03-29 01:15:49,458 - BERTopic - Cluster - Completed ✓
2026-03-29 01:15:49,458 - BERTopic - P

Validation transform complete (test set remains untouched).


In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('stopwords', quiet=True)

# Build a tokenizer aligned with BERTopic vocabulary (supports unigrams + bigrams)
stop_words = list(stopwords.words('english'))
coherence_vectorizer = CountVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b\w\w+\b'
)
coherence_analyzer = coherence_vectorizer.build_analyzer()

def preprocess_documents(doc_list):
    return [coherence_analyzer(str(doc).lower()) for doc in doc_list]

processed_train_docs = preprocess_documents(train_docs)
processed_val_docs = preprocess_documents(val_docs)

# Dictionary from train split only
id2word = Dictionary(processed_train_docs)

def topic_diversity(topics):
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    return len(unique_words) / len(all_words) if len(all_words) > 0 else 0.0

def topic_redundancy(topics):
    return 1.0 - topic_diversity(topics)

def _filter_topics_for_dictionary(topics, dictionary, min_words_per_topic=2):
    vocab = dictionary.token2id
    filtered = []
    for topic in topics:
        cleaned = []
        for term in topic:
            token = str(term).strip().lower()
            if token in vocab:
                cleaned.append(token)
        cleaned = list(dict.fromkeys(cleaned))
        if len(cleaned) >= min_words_per_topic:
            filtered.append(cleaned)
    return filtered

def coherence_score(topics, texts, dictionary, coherence_type='c_v'):
    filtered_topics = _filter_topics_for_dictionary(topics, dictionary, min_words_per_topic=2)
    if len(filtered_topics) == 0 or len(texts) == 0:
        return np.nan
    try:
        cm = CoherenceModel(
            topics=filtered_topics,
            texts=texts,
            dictionary=dictionary,
            coherence=coherence_type
        )
        value = float(cm.get_coherence())
        return value if np.isfinite(value) else np.nan
    except Exception:
        return np.nan

cv_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_v')
cv_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_v')
npmi_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_npmi')
npmi_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_npmi')

div_train = topic_diversity(bertopic_topics)
red_train = topic_redundancy(bertopic_topics)

val_outlier_ratio = float(np.mean(np.array(val_topics) == -1)) if len(val_topics) > 0 else np.nan

print(f"BERTopic Coherence C_v (train): {cv_train:.4f}")
print(f"BERTopic Coherence C_v (val):   {cv_val:.4f}")
print(f"BERTopic Coherence c_npmi (train): {npmi_train:.4f}")
print(f"BERTopic Coherence c_npmi (val):   {npmi_val:.4f}")
print(f"BERTopic Topic Diversity (train): {div_train:.4f}")
print(f"BERTopic Topic Redundancy (train): {red_train:.4f}")
print(f"Validation outlier ratio (-1 topics): {val_outlier_ratio:.4f}")
print(f"Valid topic count for coherence: {len(_filter_topics_for_dictionary(bertopic_topics, id2word))}")

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\direct_confirmation_measure.py:204: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\indirect_confirmation_measure.py:323: RuntimeWarning:

invalid value encountered in scalar divide



BERTopic Coherence C_v (train): 0.5039
BERTopic Coherence C_v (val):   nan
BERTopic Coherence c_npmi (train): 0.0962
BERTopic Coherence c_npmi (val):   inf
BERTopic Topic Diversity (train): 0.8418
BERTopic Topic Redundancy (train): 0.1582
Validation outlier ratio (-1 topics): 0.4152


Hyperparameter tuning (Train -> Validation only)

In [ ]:
from itertools import product
from sklearn.metrics.pairwise import cosine_similarity

# Precompute embeddings used in tuning
val_embeddings = sentence_model.encode(val_docs, show_progress_bar=True)

financial_keywords = {
    'inflation', 'interest', 'rate', 'rates', 'fed', 'fomc', 'earnings', 'revenue',
    'profit', 'loss', 'guidance', 'downgrade', 'upgrade', 'bank', 'credit', 'default',
    'liquidity', 'volatility', 'yield', 'bond', 'equity', 'dividend', 'margin', 'risk'
}

def assignment_confidence_metrics(prob_matrix):
    if prob_matrix is None or len(prob_matrix) == 0:
        return np.nan, np.nan
    probs = np.asarray(prob_matrix)
    if probs.ndim != 2 or probs.shape[1] == 0:
        return np.nan, np.nan
    sorted_probs = np.sort(probs, axis=1)
    top1 = sorted_probs[:, -1]
    top2 = sorted_probs[:, -2] if probs.shape[1] > 1 else np.zeros_like(top1)
    mean_top1 = float(np.mean(top1))
    mean_margin = float(np.mean(top1 - top2))
    return mean_top1, mean_margin

def inter_topic_distance(topic_model_obj):
    topic_info = topic_model_obj.get_topic_info()
    valid_topics = [int(t) for t in topic_info['Topic'].tolist() if int(t) != -1]
    if len(valid_topics) < 2 or topic_model_obj.topic_embeddings_ is None:
        return np.nan
    topic_index_map = {topic: i for i, topic in enumerate(topic_model_obj.get_topics().keys())}
    valid_idx = [topic_index_map[t] for t in valid_topics if t in topic_index_map]
    if len(valid_idx) < 2:
        return np.nan
    emb = topic_model_obj.topic_embeddings_[valid_idx]
    sim = cosine_similarity(emb)
    triu = sim[np.triu_indices_from(sim, k=1)]
    if len(triu) == 0:
        return np.nan
    value = float(np.mean(1 - triu))
    return value if np.isfinite(value) else np.nan

def financial_keyword_density(topics, keyword_set):
    all_words = [w for topic in topics for w in topic]
    if len(all_words) == 0:
        return np.nan
    hit = sum(1 for w in all_words if str(w).lower() in keyword_set)
    value = float(hit / len(all_words))
    return value if np.isfinite(value) else np.nan

def sanitize_metric(value):
    try:
        v = float(value)
    except Exception:
        return np.nan
    return v if np.isfinite(v) else np.nan

param_grid = {
    'n_neighbors': [10, 15],
    'n_components': [5, 10],
    'min_cluster_size': [10, 20],
    'min_samples': [5, 10],
    'ngram_range': [(1, 1), (1, 2)]
}

search_space = list(product(
    param_grid['n_neighbors'],
    param_grid['n_components'],
    param_grid['min_cluster_size'],
    param_grid['min_samples'],
    param_grid['ngram_range']
))

tuning_rows = []
print(f"Running {len(search_space)} hyperparameter combinations...")

for i, (n_neighbors, n_components, min_cluster_size, min_samples, ngram_range) in enumerate(search_space, start=1):
    print(f"[{i}/{len(search_space)}] n_neighbors={n_neighbors}, n_components={n_components}, min_cluster_size={min_cluster_size}, min_samples={min_samples}, ngram_range={ngram_range}")

    umap_model_i = UMAP(n_neighbors=n_neighbors, n_components=n_components, metric='cosine', random_state=42)
    hdbscan_model_i = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    vectorizer_model_i = OnlineCountVectorizer(stop_words='english', ngram_range=ngram_range)

    topic_model_i = BERTopic(
        embedding_model=sentence_model,
        umap_model=umap_model_i,
        hdbscan_model=hdbscan_model_i,
        vectorizer_model=vectorizer_model_i,
        calculate_probabilities=True,
        verbose=False
    )

    topics_train_i, _ = topic_model_i.fit_transform(train_docs, embeddings=train_embeddings)
    val_topics_i, val_probs_i = topic_model_i.transform(val_docs, embeddings=val_embeddings)

    topic_words_i = [
        [word for word, _ in topic_model_i.get_topic(t)]
        for t in topic_model_i.get_topics().keys()
        if t != -1 and topic_model_i.get_topic(t) is not None
    ]

    cv_val_i = sanitize_metric(coherence_score(topic_words_i, processed_val_docs, id2word, 'c_v'))
    npmi_val_i = sanitize_metric(coherence_score(topic_words_i, processed_val_docs, id2word, 'c_npmi'))
    div_i = sanitize_metric(topic_diversity(topic_words_i))
    red_i = sanitize_metric(topic_redundancy(topic_words_i))
    outlier_i = sanitize_metric(float(np.mean(np.array(val_topics_i) == -1)) if len(val_topics_i) > 0 else np.nan)
    mean_top1_i, mean_margin_i = assignment_confidence_metrics(val_probs_i)
    mean_top1_i = sanitize_metric(mean_top1_i)
    mean_margin_i = sanitize_metric(mean_margin_i)
    inter_dist_i = sanitize_metric(inter_topic_distance(topic_model_i))
    fin_kw_density_i = sanitize_metric(financial_keyword_density(topic_words_i, financial_keywords))
    n_topics_i = int(sum(1 for t in topic_model_i.get_topics().keys() if int(t) != -1))

    tuning_rows.append({
        'n_neighbors': n_neighbors,
        'n_components': n_components,
        'min_cluster_size': min_cluster_size,
        'min_samples': min_samples,
        'ngram_range': str(ngram_range),
        'n_topics': n_topics_i,
        'cv_val': cv_val_i,
        'npmi_val': npmi_val_i,
        'topic_diversity': div_i,
        'topic_redundancy': red_i,
        'val_outlier_ratio': outlier_i,
        'val_mean_top1_prob': mean_top1_i,
        'val_mean_margin_top1_top2': mean_margin_i,
        'inter_topic_distance': inter_dist_i,
        'financial_keyword_density': fin_kw_density_i
    })

tuning_results = pd.DataFrame(tuning_rows)

# Composite score (higher is better; outlier/redundancy are penalties)
score_cols_positive = [
    'cv_val', 'npmi_val', 'topic_diversity', 'val_mean_margin_top1_top2',
    'inter_topic_distance', 'financial_keyword_density'
 ]
score_cols_negative = ['val_outlier_ratio', 'topic_redundancy']

for col in score_cols_positive + score_cols_negative:
    values = pd.to_numeric(tuning_results[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    finite = values.dropna()
    if finite.empty:
        tuning_results[col + '_norm'] = 0.0
        continue

    if col in score_cols_positive:
        filled = values.fillna(finite.min())
    else:
        filled = values.fillna(finite.max())

    min_v, max_v = float(filled.min()), float(filled.max())
    if max_v == min_v:
        tuning_results[col + '_norm'] = 0.0
    else:
        tuning_results[col + '_norm'] = (filled - min_v) / (max_v - min_v)

# Penalize solutions with too few topics
tuning_results['topic_count_penalty'] = np.where(tuning_results['n_topics'] < 3, 0.25, 0.0)

tuning_results['composite_score'] = (
    0.25 * tuning_results['cv_val_norm'] +
    0.20 * tuning_results['npmi_val_norm'] +
    0.15 * tuning_results['topic_diversity_norm'] +
    0.10 * tuning_results['val_mean_margin_top1_top2_norm'] +
    0.10 * tuning_results['inter_topic_distance_norm'] +
    0.05 * tuning_results['financial_keyword_density_norm'] -
    0.10 * tuning_results['val_outlier_ratio_norm'] -
    0.05 * tuning_results['topic_redundancy_norm'] -
    tuning_results['topic_count_penalty']
 )

tuning_results = tuning_results.sort_values('composite_score', ascending=False).reset_index(drop=True)
best_params = tuning_results.iloc[0].to_dict()

print('\nTop 10 configurations:')
display(tuning_results.head(10))
print('\nBest configuration selected based on validation composite score:')
display(pd.DataFrame([best_params]))

Batches: 100%|██████████| 19/19 [00:01<00:00, 17.92it/s]



Running 32 hyperparameter combinations...
[1/32] n_neighbors=10, n_components=5, min_cluster_size=10, min_samples=5, ngram_range=(1, 1)


C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\direct_confirmation_measure.py:204: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\indirect_confirmation_measure.py:323: RuntimeWarning:

invalid value encountered in scalar divide



[2/32] n_neighbors=10, n_components=5, min_cluster_size=10, min_samples=5, ngram_range=(1, 2)


C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\direct_confirmation_measure.py:204: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\indirect_confirmation_measure.py:323: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning:

Mean of empty slice

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\numpy\_core\_methods.py:142: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning:

Mean of empty slice

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\numpy\_core\_methods.py:142: RuntimeWarning:

invalid value encountered in scalar divide



...

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\direct_confirmation_measure.py:204: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\indirect_confirmation_measure.py:323: RuntimeWarning:

invalid value encountered in scalar divide




Top 10 configurations:


,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,topic_redundancy,val_outlier_ratio,...,financial_keyword_density,cv_val_norm,npmi_val_norm,topic_diversity_norm,val_mean_margin_top1_top2_norm,inter_topic_distance_norm,financial_keyword_density_norm,val_outlier_ratio_norm,topic_redundancy_norm,composite_score
0,10,5,20,10,"(1, 2)",NaN,inf,0.935484,0.064516,0.261944,...,0.048387,0.0,0.0,1.000000,0.972395,0.000639,0.538312,0.000000,0.000000,0.274219
1,10,10,20,10,"(1, 2)",NaN,NaN,0.926667,0.073333,0.275124,...,0.050000,0.0,0.0,0.931299,0.971801,0.140857,0.571290,0.065041,0.068701,0.269586
2,10,5,10,5,"(1, 2)",NaN,NaN,0.895588,0.104412,0.276771,...,0.022059,0.0,0.0,0.689144,0.737638,0.912998,0.000000,0.073171,0.310856,0.245575
3,10,5,10,10,"(1, 2)",NaN,NaN,0.904918,0.095082,0.301483,...,0.026230,0.0,0.0,0.761839,0.761435,0.807725,0.085275,0.195122,0.238161,0.244035
4,15,5,20,5,"(1, 2)",NaN,inf,0.912903,0.087097,0.303130,...,0.048387,0.0,0.0,0.824057,1.000000,0.207714,0.538312,0.203252,0.175943,0.242173
5,15,5,20,5,"(1, 1)",NaN,inf,0.900000,0.100000,0.303130,...,0.064516,0.0,0.0,0.723519,1.000000,0.207714,0.868089,0.203252,0.276481,0.238554
6,15,5,20,10,"(1, 2)",NaN,NaN,0.922581,0.077419,0.360791,...,0.048387,0.0,0.0,0.899461,0.915692,0.363240,0.538312,0.487805,0.100539,0.235921
7,15,10,20,5,"(1, 2)",NaN,inf,0.926667,0.073333,0.339374,...,0.050000,0.0,0.0,0.931299,0.873930,0.214224,0.571290,0.382114,0.068701,0.235428
8,15,10,10,5,"(1, 2)",NaN,NaN,0.916176,0.083824,0.345964,...,0.026471,0.0,0.0,0.849562,0.515885,1.000000,0.090204,0.414634,0.150438,0.234548
9,10,5,20,10,"(1, 1)",NaN,inf,0.896774,0.103226,0.261944,...,0.064516,0.0,0.0,0.698384,0.972395,0.000639,0.868089,0.000000,0.301616,0.230385



Best configuration selected based on validation composite score:


,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,topic_redundancy,val_outlier_ratio,...,financial_keyword_density,cv_val_norm,npmi_val_norm,topic_diversity_norm,val_mean_margin_top1_top2_norm,inter_topic_distance_norm,financial_keyword_density_norm,val_outlier_ratio_norm,topic_redundancy_norm,composite_score
0,10,5,20,10,"(1, 2)",NaN,inf,0.935484,0.064516,0.261944,...,0.048387,0.0,0.0,1.0,0.972395,0.000639,0.538312,0.0,0.0,0.274219


Final model and test evaluation (single touch of test split)

In [21]:
from ast import literal_eval

# Refit best model on train+val, evaluate once on test
train_val_docs = train_docs + val_docs
train_val_embeddings = sentence_model.encode(train_val_docs, show_progress_bar=True)
test_embeddings = sentence_model.encode(test_docs, show_progress_bar=True)

best_umap = UMAP(
    n_neighbors=int(best_params['n_neighbors']),
    n_components=int(best_params['n_components']),
    metric='cosine',
    random_state=42
)
best_hdbscan = HDBSCAN(
    min_cluster_size=int(best_params['min_cluster_size']),
    min_samples=int(best_params['min_samples']),
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)
best_ngram = literal_eval(best_params['ngram_range'])
best_vectorizer = OnlineCountVectorizer(stop_words='english', ngram_range=best_ngram)

best_topic_model = BERTopic(
    embedding_model=sentence_model,
    umap_model=best_umap,
    hdbscan_model=best_hdbscan,
    vectorizer_model=best_vectorizer,
    calculate_probabilities=True,
    verbose=True
)

_topics_train_val, _ = best_topic_model.fit_transform(train_val_docs, embeddings=train_val_embeddings)
test_topics, test_probs = best_topic_model.transform(test_docs, embeddings=test_embeddings)

best_topic_words = [
    [word for word, _ in best_topic_model.get_topic(t)]
    for t in best_topic_model.get_topics().keys()
    if t != -1 and best_topic_model.get_topic(t) is not None
]

processed_train_val_docs = preprocess_documents(train_val_docs)
processed_test_docs = preprocess_documents(test_docs)
id2word_train_val = Dictionary(processed_train_val_docs)

cv_test = coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_v')
npmi_test = coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_npmi')
div_test = topic_diversity(best_topic_words)
red_test = topic_redundancy(best_topic_words)
test_outlier_ratio = float(np.mean(np.array(test_topics) == -1)) if len(test_topics) > 0 else np.nan
mean_top1_test, mean_margin_test = assignment_confidence_metrics(test_probs)
inter_dist_test = inter_topic_distance(best_topic_model)
fin_kw_density_test = financial_keyword_density(best_topic_words, financial_keywords)

print('Final Test Metrics (best config):')
print(f"C_v coherence (test): {cv_test:.4f}")
print(f"c_npmi coherence (test): {npmi_test:.4f}")
print(f"Topic diversity: {div_test:.4f}")
print(f"Topic redundancy: {red_test:.4f}")
print(f"Test outlier ratio (-1 topics): {test_outlier_ratio:.4f}")
print(f"Mean top-1 assignment probability: {mean_top1_test:.4f}")
print(f"Mean assignment margin (top1-top2): {mean_margin_test:.4f}")
print(f"Inter-topic distance: {inter_dist_test:.4f}")
print(f"Financial keyword density: {fin_kw_density_test:.4f}")

best_topic_model.get_topic_info().head(15)

Batches: 100%|██████████| 63/63 [00:03<00:00, 17.32it/s]
2026-03-29 01:37:30,561 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
Batches: 100%|██████████| 63/63 [00:03<00:00, 17.32it/s]
2026-03-29 01:37:30,561 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-29 01:37:39,741 - BERTopic - Dimensionality - Completed ✓
2026-03-29 01:37:39,742 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 01:37:39,741 - BERTopic - Dimensionality - Completed ✓
2026-03-29 01:37:39,742 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 01:37:39,941 - BERTopic - Cluster - Completed ✓
2026-03-29 01:37:39,943 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 01:37:39,941 - BERTopic - Cluster - Completed ✓
2026-03-29 01:37:39,943 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 01:37:40,011 - BERTopic - Representation - Co

Final Test Metrics (best config):
C_v coherence (test): nan
c_npmi coherence (test): inf
Topic diversity: 0.9444
Topic redundancy: 0.0556
Test outlier ratio (-1 topics): 0.4004
Mean top-1 assignment probability: 0.4447
Mean assignment margin (top1-top2): 0.4064
Inter-topic distance: 0.5349
Financial keyword density: 0.0333


,Topic,Count,Name,Representation,Representative_Docs
0,-1,685,-1_bank_banks_stocks_2025,"[bank, banks, stocks, 2025, td, ubs, trading, ...","[Dividend Champion, Contender, And Challenger ..."
1,0,205,0_hsbc_barclays_plc_hsbc holdings,"[hsbc, barclays, plc, hsbc holdings, bcs, hold...",[HSBC Holdings plc (HSBC) Q3 2025 Earnings Cal...
2,1,170,1_state street_state_street_mellon,"[state street, state, street, mellon, q4, york...",[The Bank of New York Mellon Corporation (BK) ...
3,2,166,2_canada_bank_bank canada_td,"[canada, bank, bank canada, td, royal bank, ro...",[The Toronto-Dominion Bank (TD:CA) Q4 2025 Ear...
4,3,95,3_deutsche_deutsche bank_bank_db,"[deutsche, deutsche bank, bank, db, bank db, d...",[Deutsche Bank Aktiengesellschaft (DB) Q3 2025...
5,4,93,4_fund_commentary_2025 commentary_2025,"[fund, commentary, 2025 commentary, 2025, q2 2...","[Templeton Foreign Fund Q2 2025 Commentary, Te..."
6,5,93,5_ubs_ubs group_years_group,"[ubs, ubs group, years, group, invested, years...",[Here's How Much $1000 Invested In UBS Gr 5 Ye...
7,6,91,6_santander_banco_banco santander_tsb,"[santander, banco, banco santander, tsb, san, ...",[Market Chatter: Banco Santander Reaches Out t...
8,7,80,7_dividend_dividend stocks_stocks_etfs,"[dividend, dividend stocks, stocks, etfs, high...",[Our Top 10 High Growth Dividend Stocks - Sept...
9,8,78,8_bnp_paribas_bnp paribas_stab,"[bnp, paribas, bnp paribas, stab, new issues, ...",[CROWN - BNP Paribas Primary New Issues: NO ST...
